In [1]:
# importing Database

import os

from dotenv import load_dotenv

from langchain_community.utilities import SQLDatabase

load_dotenv()

mysql_uri = (
    f"mysql+pymysql://"
    f"{os.getenv('MYSQL_USER')}:"
    f"{os.getenv('MYSQL_PASSWORD')}@"
    f"{os.getenv('MYSQL_HOST')}:"
    f"{os.getenv('MYSQL_PORT')}/"
    f"{os.getenv('MYSQL_DATABASE')}"
)

db = SQLDatabase.from_uri(mysql_uri)

print("Dialect:", db.dialect)
print("Tables:", db.get_usable_table_names())

C:\Users\jassi\AppData\Local\Temp\ipykernel_18480\1424904557.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Dialect: mysql
Tables: ['2017_budgets', 'customers', 'products', 'regions', 'sales_order', 'sales_orders_old', 'state_regions']


In [2]:
# adding sqlite to save checkpoints

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect(
    'checkpoints.db',
    check_same_thread=False
)

checkpoint = SqliteSaver(conn)

In [3]:
# importing llm

# from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
)

In [4]:
# importing SQL Tools

from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(
    db = db,
    llm = llm
)

tools = toolkit.get_tools()

In [5]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print()

sql_db_query
Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema
Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables
Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker
Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [6]:
system_prompt = f"""
You are an agent designed to interact with a MySQL database.

Given a user's question, determine the relevant tables and columns,
generate a syntactically correct MySQL SQL query, execute it, and
return a clear natural-language answer.

Rules:

1. Always inspect the available tables first.
2. Retrieve the schema of relevant tables before generating SQL.
3. Only query tables that are relevant to the user's question.
4. Only select columns that are necessary.
5. Always check the SQL query before executing it.
6. If the SQL query produces an error, analyze the error and fix the query.
7. NEVER execute INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE,
   CREATE, or other data-modifying statements.
8. Only perform read-only queries.
9. Do not expose unnecessary database details to the user.
10. Return the final answer in clear natural language.

11. After the SQL query has been checked and approved,
    ALWAYS execute the query using sql_db_query before producing
    the final answer.

12. NEVER infer numerical results from sample rows returned by
    sql_db_schema. Numerical answers must come from executing SQL.

13. Do not answer a database question using information from the
    schema sample rows when the answer can be obtained by executing SQL.

Database dialect: {db.dialect}
"""

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model = llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=checkpoint
)

In [9]:
config = {
    'configurable': {
        'thread_id': 'user_1'
    }
}

In [77]:
question = "What was the total education budget in 2017?"

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    },
    config=config
)

In [78]:
question = "What was the total sales amount in the first month of sales_orders_old?"

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    },
    config=config
)

In [79]:
# for event in agent.stream(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": "What was the total sales amount in the first month of sales_orders_old?"
#             }
#         ]
#     },
#     stream_mode="updates",
#     config=config
# ):
#     for node, data in event.items():

#         print(f"\n===== {node} =====")

#         if "messages" in data:
#             for message in data["messages"]:
#                 print(message)

In [10]:
state = agent.get_state(config)

for msg in state.values["messages"]:
    if msg.type in ["human", "ai"]:
        print(f"\n{msg.type.upper()}:")
        print(msg.content)


HUMAN:
What was the total education budget in 2017?

AI:


AI:


AI:


AI:


AI:


AI:


AI:


AI:
The database only contains a table called **2017_budgets**, which lists budget amounts for various products. Summing all of those amounts gives a total of **62,700,262.34** (in the units used by the table).  

If you were looking for a specific “education” budget, that information isn’t present in the current schema.

HUMAN:
What was the total sales amount in the first month of sales_orders_old?

AI:


AI:


AI:


AI:
The total sales amount for the first month recorded in the `sales_orders_old` table (January 2014) is **$25,020,272.30**.

HUMAN:
What was the total education budget in 2017?

AI:


AI:
I’m sorry, but the database doesn’t contain any information labeled “education” in the 2017 budgets. The only budget data available is the overall totals for the products listed in the `2017_budgets` table, and there’s no separate entry for an education budget.

HUMAN:
What was the total sal

In [ ]:
def delete_chat(thread_id):
    """Delete all saved checkpoint data for a specific thread."""
    
    conn.execute(
        "DELETE FROM checkpoints WHERE thread_id = ?",
        (thread_id,)
    )
    
    conn.execute(
        "DELETE FROM writes WHERE thread_id = ?",
        (thread_id,)
    )
    
    conn.commit()
    
    print(f"Thread '{thread_id}' deleted successfully.")

In [12]:
config2 = {
    'configurable': {
        'thread_id': 'user_2'
    }
}

In [15]:
question = 'What is the total amount of products?'

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    },
    config=config2
)

{'messages': [HumanMessage(content='What is the total amount of sale?', additional_kwargs={}, response_metadata={}, id='f80f477b-6dd2-4251-b1dc-fd2b6d803e2f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to find tables and columns. Let's list tables.", 'tool_calls': [{'id': 'fc_436b7293-ca6e-4918-8f33-1be5f758ef09', 'function': {'arguments': '{"tool_input":""}', 'name': 'sql_db_list_tables'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 671, 'total_tokens': 710, 'completion_time': 0.041074443, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.248654146, 'prompt_tokens_details': {'cached_tokens': 512}, 'queue_time': 0.342784599, 'total_time': 0.289728589}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_99996fee8e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a085c4-9915-7c80-943a-0e54d59821a1-0

In [20]:
state = agent.get_state(config)

for msg in state.values["messages"]:
    if msg.type in ["human", "ai"]:
        print(f"\n{msg.type.upper()}:")
        print(msg.content)


HUMAN:
What was the total education budget in 2017?

AI:


AI:


AI:


AI:


AI:


AI:


AI:


AI:
The database only contains a table called **2017_budgets**, which lists budget amounts for various products. Summing all of those amounts gives a total of **62,700,262.34** (in the units used by the table).  

If you were looking for a specific “education” budget, that information isn’t present in the current schema.

HUMAN:
What was the total sales amount in the first month of sales_orders_old?

AI:


AI:


AI:


AI:
The total sales amount for the first month recorded in the `sales_orders_old` table (January 2014) is **$25,020,272.30**.

HUMAN:
What was the total education budget in 2017?

AI:


AI:
I’m sorry, but the database doesn’t contain any information labeled “education” in the 2017 budgets. The only budget data available is the overall totals for the products listed in the `2017_budgets` table, and there’s no separate entry for an education budget.

HUMAN:
What was the total sal

In [ ]:
delete_chat(config2['configurable']['thread_id'])

Thread 'user_2' deleted successfully.


In [23]:
def get_chat_history():
    rows = conn.execute(
        "SELECT DISTINCT thread_id FROM checkpoints"
    ).fetchall()

    chat_history = {
        "threads": {}
    }

    for (thread_id,) in rows:
        config = {
            "configurable": {
                "thread_id": thread_id
            }
        }

        state = agent.get_state(config)

        messages = []

        for msg in state.values["messages"]:
            if msg.type == "human" and msg.content:
                messages.append({
                    "role": "user",
                    "content": msg.content
                })

            elif msg.type == "ai" and msg.content:
                messages.append({
                    "role": "assistant",
                    "content": msg.content
                })

        chat_history["threads"][thread_id] = messages

    return chat_history

In [24]:
chat_history = get_chat_history()

print(chat_history)

{'threads': {'user_1': [{'role': 'user', 'content': 'What was the total education budget in 2017?'}, {'role': 'assistant', 'content': 'The database only contains a table called **2017_budgets**, which lists budget amounts for various products. Summing all of those amounts gives a total of **62,700,262.34** (in the units used by the table).  \n\nIf you were looking for a specific “education” budget, that information isn’t present in the current schema.'}, {'role': 'user', 'content': 'What was the total sales amount in the first month of sales_orders_old?'}, {'role': 'assistant', 'content': 'The total sales amount for the first month recorded in the\u202f`sales_orders_old`\u202ftable (January\u202f2014) is **$25,020,272.30**.'}, {'role': 'user', 'content': 'What was the total education budget in 2017?'}, {'role': 'assistant', 'content': 'I’m sorry, but the database doesn’t contain any information labeled “education” in the 2017 budgets. The only budget data available is the overall total

In [1]:
def invoke_Agent(config):
    agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    },
    config=config
    )

    return 'message sent succesfully'

    